# Configure Codex for regulated enterprise environments

Financial services, healthcare, public-sector, and other regulated organizations need more than a filesystem policy when they introduce an AI coding assistant. They need an operating model that covers identity, human oversight, permitted execution modes, network access, integrations, sensitive data, monitoring, change management, and device deployment.

This cookbook provides a general-purpose starting point for the latest supported local Codex desktop, CLI, and IDE clients. It includes one complete enterprise `requirements.toml`, one matching client `config.toml`, managed model and identity governance, enterprise configuration guidance, and a practical rollout checklist.

The examples are technical deployment guidance, not a compliance certification, legal interpretation, contractual data-processing commitment, or substitute for an organization-specific risk assessment.

## What you will learn

By the end of this cookbook, you will be able to:

- Distinguish enforced enterprise requirements from changeable client defaults.
- Configure one shared `requirements.toml` and one matching `config.toml`.
- Govern identity, approvals, execution, networking, integrations, secrets, and observability.
- Restrict visible models and apply Windows or macOS device controls.
- Deploy, validate, and support a regulated enterprise rollout.

## Before you begin

To follow this cookbook, you need:

- An approved ChatGPT Enterprise or regulated-workspace deployment and the authority to configure its managed Codex policies.
- The latest approved local Codex desktop, CLI, or IDE release across all managed devices.
- The appropriate enterprise identity, device-management, security, and change-management approvals.

Review the example files locally. Model-catalog export is a separate action that uses an already authorized Codex account.

## Download the starter files

Use these two reviewed reference files as the starting point for an enterprise deployment:

- [Download the enterprise requirements.toml](regulated_industry_configuration/requirements.toml) - The managed policy that defines supported, non-overridable boundaries.
- [Download the matching config.toml](regulated_industry_configuration/config.toml) - The recommended client defaults used inside those boundaries.

Keep the two TOML files paired. Do not assume that a setting becomes enforced simply because it appears in client configuration or managed defaults.

For operating-system settings, see [Configure Windows and macOS deployment](#configure-windows-and-macos-deployment).

## Follow the deployment process

1. Define the organization's risk groups, identity boundaries, and required human oversight.
2. Review `requirements.toml` and pair it with a compatible `config.toml`.
3. Configure enterprise authentication, approved models, and platform-specific controls.
4. Distribute both files through supported enterprise or device management.
5. Verify the effective policy, run a limited pilot, and collect support feedback.

## Separate enterprise requirements from client defaults

Codex configuration has two layers:

| Layer | File | Purpose | Can the user override it? |
| --- | --- | --- | --- |
| Enterprise requirements | `requirements.toml` | Define supported, enforced policy boundaries. | No, when delivered through a supported managed-requirements source. |
| Client configuration | `config.toml` | Choose client behavior, defaults, and platform settings. | Yes, unless a matching requirement or external control prevents it. |

Managed settings delivered through `managed_config.toml` or macOS mobile device management (MDM) remain defaults unless they are also enforced by a supported requirement.

For example, a requirement can permit cached or disabled web search while `config.toml` selects cached search. The client cannot select live search. Similarly, an `on-request` approval policy still permits eligible user-approved exceptions; a no-exception deployment needs matching `never` settings.

Identity-provider policies, endpoint security, firewall rules, model entitlements, retention, and data residency require their corresponding enterprise controls. TOML alone does not establish those boundaries.

Use the latest release approved by the organization's device-management process. Where an approved installation uses npm:

```bash
npm install -g @openai/codex@latest
```

Validate settings against the installed client's current schema. With managed permission profiles, define filesystem and command-network boundaries in the selected profile instead of mixing in older `sandbox_mode` or `[sandbox_workspace_write]` settings.

## Define the regulated operating model

Review the baseline against these governance areas:

- **Identity and models:** Enterprise sign-in, verified workspace access, approved model catalogs, and backend entitlements.
- **Human oversight and execution:** Reviewed permission profiles, approval policies, command rules, and protected production workflows.
- **Networking and integrations:** Controlled command networking, web search, apps, Model Context Protocol (MCP) servers, and plugins.
- **Sensitive data and observability:** Protected credentials, reduced local retention, approved telemetry, and incident feedback.
- **Managed devices:** Elevated Windows sandboxing, private-desktop isolation, macOS MDM, and documented compatibility exceptions.

The example supports supervised engineering. Organizations that cannot permit user-approved exceptions need the stricter approval model described below.

## Create the enterprise requirements file

The complete managed policy is available as [requirements.toml](regulated_industry_configuration/requirements.toml):

```toml
# Cross-platform regulated-enterprise requirements for the latest Codex release.
# Deploy through enterprise-managed configuration or the system policy path.
# These requirements establish governance boundaries, not user defaults.

allowed_approval_policies = ["on-request", "untrusted"]
allowed_approvals_reviewers = ["user"]
allowed_web_search_modes = ["disabled", "cached"]
default_permissions = "regulated_workspace"
allow_login_shell = false
allow_managed_hooks_only = true
allow_appshots = false
allow_remote_control = false

# Optionally enforce an approved model catalog installed on each managed device.
# model_catalog_json = "/etc/codex/approved-models.json"

# Explicit empty allowlists block standalone and plugin-bundled MCP servers.
# Add only approved, verified server and plugin identities.
mcp_servers = {}
plugins = {}

# Enforce the stronger sandbox and isolated desktop on native Windows clients.
[windows]
allowed_sandbox_implementations = ["elevated"]
sandbox_private_desktop = true

[allowed_permission_profiles]
":read-only" = true
regulated_read_only = true
regulated_workspace = true

[permissions.regulated_read_only]
description = "Inspect approved files without modifying the workspace or using command networking."
extends = ":read-only"

[permissions.regulated_read_only.network]
enabled = false

[permissions.regulated_workspace]
description = "Use the standard workspace sandbox with human oversight, protected secrets, and restricted command networking."
extends = ":workspace"

[permissions.regulated_workspace.filesystem]
glob_scan_max_depth = 6

[permissions.regulated_workspace.filesystem.":workspace_roots"]
".env" = "deny"
".env.local" = "deny"
"**/.env" = "deny"
"**/.env.*" = "deny"
"**/*.env" = "deny"
"**/*.key" = "deny"
"**/*.pem" = "deny"
"**/*.p12" = "deny"
"**/*.pfx" = "deny"

[permissions.regulated_workspace.network]
enabled = false

# Protect sensitive credentials across approved profiles, not only inside roots.
# Native Windows shell reads also require operating-system and endpoint controls.
[permissions.filesystem]
deny_read = [
  "~/.ssh/id_rsa",
  "~/.ssh/id_ed25519",
  "~/.aws/credentials",
  "~/.azure",
  "~/.config/gcloud",
  "~/.kube/config",
  "~/.docker/config.json",
  "~/.npmrc",
  "~/.pypirc",
  "~/.netrc",
  "/**/.env",
  "/**/*.env",
  "/**/*.pem",
  "/**/*.key",
  "/**/*.p12",
  "/**/*.pfx",
]

[features]
browser_use = false
browser_use_external = false
computer_use = false
enable_mcp_apps = false
guardian_approval = false
in_app_browser = false
memories = false
memory_tool = false
plugin_sharing = false
remote_control = false
remote_plugin = false
skill_mcp_dependency_install = false

# Prompt for reviewable engineering actions and forbid high-risk operations.
[rules]
prefix_rules = [
  { pattern = [{ any_of = ["bash", "sh", "zsh", "pwsh", "powershell", "cmd"] }], decision = "prompt", justification = "Nested shell entry points require explicit human review." },
  { pattern = [{ token = "rm" }, { any_of = ["-rf", "-fr", "-Rf", "-fR"] }], decision = "forbidden", justification = "Forced recursive deletion is not allowed." },
  { pattern = [{ token = "Remove-Item" }, { token = "-Recurse" }, { token = "-Force" }], decision = "forbidden", justification = "Forced recursive deletion is not allowed." },
  { pattern = [{ token = "Remove-Item" }, { token = "-Force" }, { token = "-Recurse" }], decision = "forbidden", justification = "Forced recursive deletion is not allowed." },
  { pattern = [{ any_of = ["rm", "rmdir", "del", "Remove-Item"] }], decision = "prompt", justification = "A human must review destructive filesystem operations." },
  { pattern = [{ token = "git" }, { token = "reset" }, { token = "--hard" }], decision = "forbidden", justification = "A hard reset can discard uncommitted work." },
  { pattern = [{ token = "git" }, { any_of = ["commit", "push", "clean", "rebase", "checkout", "switch"] }], decision = "prompt", justification = "A human must approve repository mutations and publication." },
  { pattern = [{ any_of = ["curl", "wget", "Invoke-WebRequest", "Invoke-RestMethod"] }], decision = "prompt", justification = "A human must review attempted data transfer." },
  { pattern = [{ any_of = ["npm", "pnpm", "yarn", "pip", "pip3", "uv"] }, { any_of = ["install", "add"] }], decision = "prompt", justification = "Dependency changes require approved package sources and human review." },
  { pattern = [{ any_of = ["docker", "podman"] }, { any_of = ["run", "pull", "build", "push"] }], decision = "prompt", justification = "Container execution and registry changes require human review." },
  { pattern = [{ any_of = ["ngrok", "localtunnel", "devtunnel"] }], decision = "forbidden", justification = "Public tunneling is not allowed; use an approved internal environment." },
  { pattern = [{ token = "cloudflared" }, { token = "tunnel" }], decision = "forbidden", justification = "Public tunneling is not allowed; use an approved internal environment." },
  { pattern = [{ token = "code" }, { token = "tunnel" }], decision = "forbidden", justification = "Editor tunnels are not allowed without an approved exception." },
  { pattern = [{ token = "ssh" }, { any_of = ["-R", "-L", "-D"] }], decision = "forbidden", justification = "SSH port forwarding and tunnels require an approved external workflow." },
  { pattern = [{ any_of = ["sudo", "su", "runas", "pkexec"] }], decision = "forbidden", justification = "Privilege escalation is not allowed from agent-managed sessions." },
  { pattern = [{ any_of = ["mimikatz", "procdump", "procdump.exe"] }], decision = "forbidden", justification = "Credential extraction and process-memory dumping are not allowed." },
  { pattern = [{ any_of = ["reg", "reg.exe"] }, { any_of = ["add", "delete", "import"] }], decision = "forbidden", justification = "Windows registry changes require approved endpoint-management workflows." },
  { pattern = [{ any_of = ["sc", "sc.exe"] }, { any_of = ["create", "config", "delete"] }], decision = "forbidden", justification = "Windows service changes require approved endpoint-management workflows." },
  { pattern = [{ token = "kubectl" }, { any_of = ["apply", "delete", "patch", "exec", "edit", "port-forward"] }], decision = "forbidden", justification = "Production changes and service exposure require approved deployment pipelines." },
  { pattern = [{ token = "terraform" }, { any_of = ["apply", "destroy"] }], decision = "forbidden", justification = "Infrastructure changes require approved change management." },
  { pattern = [{ token = "gh" }, { token = "pr" }, { token = "merge" }], decision = "forbidden", justification = "Pull-request merges remain human-controlled actions outside Codex." },
]
```

### Explain the organization-wide requirements

Keep top-level settings before the first TOML table header.

| Managed setting | Purpose |
| --- | --- |
| `allowed_approval_policies` and `allowed_approvals_reviewers` | Limit execution to supervised workflows reviewed by a human user. |
| `allowed_web_search_modes` | Permit disabled or cached search, but not live search. |
| `default_permissions` | Start with the managed `regulated_workspace` permission profile. |
| `allow_login_shell` and `allow_managed_hooks_only` | Reject login shells and skip unmanaged hooks. |
| `allow_appshots` and `allow_remote_control` | Disable Appshots and supported remote-control features. |
| `[windows]` | Require the elevated native Windows sandbox and private desktop. |
| `mcp_servers = {}` and `plugins = {}` | Block standalone and plugin-bundled MCP servers until explicitly approved. |
| `model_catalog_json`, when enabled | Enforce a protected local model catalog without changing backend entitlements. |

An explicit empty MCP or plugin allowlist is different from omitting the requirement. Keep each empty until the organization approves the integration's identity, access, and ownership.

### Explain the permission profiles

> **Beta:** Permission profiles are under active development. Verify support against the latest approved client and current configuration reference.

`[allowed_permission_profiles]` permits `:read-only` and two enterprise profiles. Omitted profiles, including unrestricted full access, are unavailable.

`regulated_read_only` extends `:read-only` for investigation without workspace writes or command networking. `regulated_workspace` extends `:workspace`, protects common environment, key, and certificate files, and disables sandboxed-command networking. Neither network setting disconnects the authenticated Codex client.

`glob_scan_max_depth = 6` limits recursive deny-pattern expansion. Wildcard coverage can vary by platform and startup state. Global `[permissions.filesystem].deny_read` also protects SSH, cloud, Kubernetes, container, package-manager, and certificate credentials. On native Windows, shell subprocess reads require additional operating-system or endpoint controls.

### Explain feature restrictions

The managed `[features]` table disables:

- Interactive browser surfaces and external browsing.
- Device interaction, remote control, and persistent memory.
- Unreviewed MCP apps, plugins, sharing, and dependency installation.
- Automated approval through `guardian_approval = false` when human review is required.

Verify each feature on the deployed management surface. For example, `plugin_sharing` applies to supported cloud-managed requirements.

### Explain command-review rules

`[rules].prefix_rules` either prompts a human or forbids a matching action:

- `prompt` covers nested shells, file deletion, Git changes, data transfer, dependency installation, and container operations.
- `forbidden` covers forced recursive deletion, destructive Git resets, tunneling, privilege escalation, credential extraction, protected infrastructure changes, and pull-request merges.

The nested-shell rule includes `bash`, `sh`, `zsh`, `pwsh`, `powershell`, and `cmd`. Commands using variable expansion, redirection, or control flow can be evaluated as one shell invocation. Disabling login shells does not replace nested-shell approval.

Prefix rules supplement sandboxing, endpoint security, identity policy, and repository protections; they cannot inspect every indirect execution path.

## Create the matching client configuration

The complete client configuration is available as [config.toml](regulated_industry_configuration/config.toml):

```toml
#:schema https://developers.openai.com/codex/config-schema.json
# Cross-platform device or user defaults for the latest Codex release.
# Pair this file with the corresponding enforced requirements.toml.

approval_policy = "on-request"
approvals_reviewer = "user"
default_permissions = "regulated_workspace"
web_search = "cached"
allow_login_shell = false
model_reasoning_effort = "medium"

# Configure a default model only after verifying its enterprise entitlement;
# install the optional catalog and enforce it through managed requirements.
# model_catalog_json = "/etc/codex/approved-models.json"

forced_login_method = "chatgpt"
cli_auth_credentials_store = "keyring"
mcp_oauth_credentials_store = "keyring"

# Match the managed native Windows sandbox and isolated-desktop requirements.
[windows]
sandbox = "elevated"
sandbox_private_desktop = true

[apps._default]
enabled = false
destructive_enabled = false
open_world_enabled = false

[analytics]
enabled = false

[feedback]
enabled = true

[history]
persistence = "none"

[shell_environment_policy]
inherit = "core"
ignore_default_excludes = false

[shell_environment_policy.filters]
"*PASSWORD*" = "exclude"
"*CREDENTIAL*" = "exclude"
"*PRIVATE*" = "exclude"

[otel]
environment = "regulated-production"
log_user_prompt = false
```

### Explain everyday operating defaults

Each client setting must remain inside its corresponding managed requirement.

| Client setting | Starting behavior |
| --- | --- |
| `approval_policy = "on-request"` | Request human review before eligible sensitive actions. |
| `approvals_reviewer = "user"` | Send approval decisions to the human user. |
| `default_permissions = "regulated_workspace"` | Start inside the managed workspace profile. |
| `web_search = "cached"` | Avoid live web retrieval. |
| `allow_login_shell = false` | Disable login-shell behavior. |
| `model_reasoning_effort = "medium"` | Choose a balanced reasoning default without granting model access. |

`on-request` supports supervised exceptions. When exceptions must be impossible, select `never` in both the managed allowlist and client configuration.

### Explain identity and credential storage

`forced_login_method = "chatgpt"` selects enterprise ChatGPT authentication. Apply it through protected managed configuration when users must not change it, and enforce account boundaries through enterprise identity controls.

If workspace binding is necessary, add `forced_chatgpt_workspace_id` only after verifying the actual organization workspace ID. The two `keyring` settings request operating-system credential storage; verify endpoint support before rollout.

### Explain apps, retention, feedback, and telemetry

`[apps._default]` disables apps and destructive or open-world actions. `[analytics].enabled = false` reduces optional analytics, while `[history].persistence = "none"` disables local session history. These defaults do not change contractual retention or enterprise audit obligations.

`[feedback].enabled = true` keeps `/feedback` available for support. `[otel]` labels approved telemetry and disables prompt logging without configuring an exporter or external destination.

### Explain shell environment filtering

`[shell_environment_policy].inherit = "core"` reduces inherited variables. Built-in exclusions protect names containing `KEY`, `SECRET`, and `TOKEN`; the additional filters cover `PASSWORD`, `CREDENTIAL`, and `PRIVATE`.

Do not combine these filters with older `exclude` or `include_only` arrays in the same configuration layer.

### Match requirements to client defaults

Keep approval policy, reviewer, permission profile, web-search mode, login-shell behavior, and Windows sandbox settings aligned across both files. Identity enforcement, model authorization, local privacy, and observability still require the corresponding managed or external enterprise control.

## Restrict visible models and reasoning options

Use the control that matches the intended boundary:

- `model` and `model_reasoning_effort` select client defaults.
- Managed `model_catalog_json` limits the models and reasoning options visible in Codex.
- Workspace entitlements and backend authorization determine actual model access.

Set `model_catalog_json` as a top-level managed requirement before any TOML table header. On macOS:

```toml
model_catalog_json = "/etc/codex/approved-models.json"
```

On native Windows, use a protected local device path:

```toml
model_catalog_json = 'C:\ProgramData\OpenAI\Codex\approved-models.json'
```

Cloud management can assign the catalog policy to enterprise groups, but device management must still install and protect the local JSON file. An HTTPS URL does not distribute or load it.

Optional client defaults can select an approved reasoning level and the same protected catalog:

```toml
model_reasoning_effort = "medium"
model_catalog_json = "/etc/codex/approved-models.json"
```

When a model default is required, choose its identifier from the organization's approved catalog and verify the account entitlement. On Windows, replace the catalog path with the protected Windows path above.

### Apply different model policies to enterprise groups

Assign a reviewed catalog and matching managed policy to each risk group. Verify the intended users receive that policy, other groups retain their expected access, and system, cloud, or MDM sources compose correctly.

### Build an approved model catalog

Export the catalog from an authorized account:

```bash
codex debug models > approved-models.json
```

Preserve each approved model's complete metadata, remove unapproved models and reasoning levels, protect the installed JSON file, and restart Codex after changes. Test the result with a pilot group.

A local catalog controls the Codex model picker, not backend authorization. Enforce actual model access through the appropriate enterprise entitlement or server-side control.

## Review important enterprise security decisions

### Choose an approval operating model

| Operating model | Managed policy | Result |
| --- | --- | --- |
| Supervised engineering | `on-request` or `untrusted` with reviewer `user`. | A human can approve eligible exceptions. |
| No user-approved exceptions | `never` with matching client configuration. | Actions requiring approval are rejected. |

`on-request` can allow execution outside the normal sandbox after user approval. If that is unacceptable, set `allowed_approval_policies = ["never"]`, select `approval_policy = "never"`, and replace relevant `prompt` rules with `forbidden`.

### Protect identity, integrations, and networking

Bind access to the organization's verified identity provider and ChatGPT workspace. Add a workspace identifier only after administrators confirm it independently.

Command-network restrictions and cached search do not disconnect Codex from its authenticated service. Govern client connections through identity, proxy, and egress controls. Approve MCP servers, apps, internal services, and any network-enabled permission profiles individually.

### Review retention, telemetry, and residency

Disabled local history can complicate incident investigation. Maintain the enterprise audit process required by the organization's retention policy.

Approve telemetry destinations, authentication, classification, retention, and access before configuring an exporter. Handle data residency through the appropriate supported administrative and contractual controls, not an invented TOML value.

### Define the filesystem boundary

The workspace profile supports development while denying common credential files. A selected workspace remains trusted by that profile, so prevent prohibited network shares or directories from becoming workspaces through endpoint, identity, operating-system, or file-share policy.

On native Windows, managed `deny_read` protects direct file tools; shell subprocess reads need separate endpoint or operating-system controls. Wildcard coverage depends on matching files and scan depth. Use a stricter custom profile when explicit readable roots are required.

## Configure Windows and macOS deployment

Deliver enforced requirements through supported cloud, system, or device management. Use managed defaults only for starting values, and verify source precedence before broad deployment.

### Apply native Windows sandbox settings

The shared `requirements.toml` enforces the stronger native Windows sandbox and isolated desktop:

```toml
[windows]
allowed_sandbox_implementations = ["elevated"]
sandbox_private_desktop = true
```

The matching `config.toml` selects that implementation:

```toml
[windows]
sandbox = "elevated"
sandbox_private_desktop = true
```

`elevated` does not grant the agent unrestricted administrator access. The elevated-only allowlist prevents an unapproved `unelevated` fallback, and the managed private-desktop requirement preserves user-interface isolation.

Install system-managed Windows requirements at `%ProgramData%\OpenAI\Codex\requirements.toml` or distribute supported cloud-managed requirements. Protect the active policy from user modification.

If endpoint restrictions prevent the elevated sandbox from initializing, document the exception, assign a remediation owner, and approve compensating controls before changing both files:

```toml
# requirements.toml - reviewed compatibility exception
[windows]
allowed_sandbox_implementations = ["unelevated"]
sandbox_private_desktop = true
```

```toml
# config.toml - matching compatibility exception
[windows]
sandbox = "unelevated"
sandbox_private_desktop = true
```

Allow both implementations, or disable the private desktop, only after a separate security review.

### Apply macOS managed configuration

macOS uses the native Seatbelt sandbox and does not require a separate `[macos]` table:

- `/etc/codex/requirements.toml` provides system-managed requirements.
- `/etc/codex/managed_config.toml` provides managed defaults.
- `~/.codex/config.toml` contains user preferences inside the approved boundary.

Mobile device management (MDM) uses the `com.openai.codex` preference domain:

- `requirements_toml_base64` supplies enforced requirements.
- `config_toml_base64` supplies managed defaults.

MDM defaults take precedence over system-managed defaults and user settings. Requirements follow their own documented source precedence.

### Compare platform-specific deployment

| Concern | Native Windows | macOS |
| --- | --- | --- |
| Sandbox | Elevated implementation and private desktop. | Native Seatbelt sandbox. |
| System requirements | `%ProgramData%\OpenAI\Codex\requirements.toml`. | `/etc/codex/requirements.toml` or MDM. |
| Managed defaults | Supported Windows managed configuration. | `/etc/codex/managed_config.toml` or MDM. |
| Exceptions | Security-reviewed `unelevated` fallback. | Supported device-management and endpoint controls. |

## Validate the deployment and rollout

Inspect the active managed policy on each device; reviewing example files alone does not prove enforcement.

1. Confirm every local client uses the latest approved release.
2. Open `/debug-config` and verify the requirements source, approval settings, permission profile, and platform controls.
3. Confirm enterprise authentication, workspace, credential storage, integrations, and any model catalog.
4. Check search, command networking, analytics, local history, and approved telemetry.
5. Verify sensitive actions prompt for review and prohibited actions are blocked.
6. On Windows, confirm the elevated sandbox and private desktop, or document an approved exception.
7. On macOS, confirm the system or MDM policy source, defaults precedence, and Seatbelt enforcement.
8. If troubleshooting is necessary, run `/feedback` and share the feedback ID through approved support channels.

Do not include credentials, customer data, confidential code, or other sensitive material in support requests.

## Adapt the baseline to the organization

Start with one managed baseline and create reviewed variations for groups with different risk profiles. For example, engineering can use supervised workspace access, while production support can require read-only permissions and a no-exception approval policy.

Keep both shared TOML files synchronized, verify active management sources, and adapt identity, networking, data handling, and endpoint controls to the organization's requirements.

## References

- [Managed configuration and enterprise requirements](https://learn.chatgpt.com/docs/enterprise/managed-configuration)
- [Enterprise login and workspace authentication controls](https://learn.chatgpt.com/docs/auth#enforce-a-login-method-or-workspace)
- [Workspace model availability and administrative controls](https://learn.chatgpt.com/docs/enterprise/workspace-model-availability)
- [Permission profiles and execution boundaries](https://learn.chatgpt.com/docs/permissions)
- [Agent approvals, sandboxing, and security](https://learn.chatgpt.com/docs/agent-approvals-security)
- [Native Windows sandbox](https://learn.chatgpt.com/docs/windows/windows-sandbox)
- [Configuration reference](https://learn.chatgpt.com/docs/config-file/config-reference)
- [Advanced configuration and shell environment policy](https://learn.chatgpt.com/docs/config-file/config-advanced)
- [Managed execution rules](https://learn.chatgpt.com/docs/agent-configuration/rules)